In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# pip install matplotlib scikit-learn torchdiffeq numpy==2.0.0 jupyterplot siren_pytorch pinnstorch lightning pandas seaborn

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import wandb
import os
from torch import nn
import warnings
warnings.filterwarnings('ignore')

# os.environ["WANDB_NOTEBOOK_NAME"] = "PINN_GPE.ipynb"
wandb.login(host='http://localhost:8080', 
            key='local-0f9731927571d8659bd45b0d14c25d90e730aa0f')
run = wandb.init(
    project="PINN_0",
    settings=wandb.Settings(
        base_url="http://localhost:8080"
    )
)
# run = wandb.init(mode="offline", project="PINN2")
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
np.random.seed(0xFA1AFE1)

data = np.load('data_demo/GPE_phase_PHASE_3D_10x10x10_R_7067c_BACKUP_PRESENT_0_intermediate_0.npz')

wandb: Loaded settings from
wandb:   /Users/tarkhov/.config/wandb/settings
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for http://localhost:8080.
wandb: Appending key for localhost:8080 to your netrc file: /Users/tarkhov/.netrc
wandb: Currently logged in as: tarkhov to http://localhost:8080. Use `wandb login --relogin` to force relogin


In [4]:
try:
    if torch.cuda.is_available():
        device = 'cpu:0'
    else:
        device = 'cuda:0'
except:
    device = 'cpu:0'

In [5]:
beta = 1.
step = 0.001
V = 1000

In [6]:
from DGPE.GPElib.lyapunov_generator import LyapunovGenerator
from DGPE.GPElib.dynamics_generator import DynamicsGenerator

dgpe = DynamicsGenerator(N_part_per_well=1.,
                         W=0, disorder_seed=53,
                         N_wells=(10,10,10), dimensionality=3, anisotropy=1.,
                         threshold_XY_to_polar=0.25,
                         J=1, beta=beta,  #beta_disorder_amplitude=2.,
                        integration_method='RK45',
                         rtol=1e-9, atol=1e-9,
                        # smooth_quench=True,
                         smooth_quench_to_room=True,
                         reset_steps_duration=5,
                         calculation_type='lyap_save_all',
                        integrator='scipy',
                        # gpu_integrator='torch',
                        # gpu_id=0,
                         time=5, step=step, t_steps=5000, gamma=1.,
                         quenching_gamma=1.)

e = 6
# dgpe.X, dgpe.Y = data['order_parameters'][0,e,:,:,:,:].real, data['order_parameters'][0,e,:,:,:,:].imag
dgpe.X.shape, dgpe.Y.shape
dgpe.set_init_XY(data['order_parameters'][0,e,:,:,:,0].real, data['order_parameters'][0,e,:,:,:,0].imag)
dgpe.step = step
dgpe.n_steps = 5000
dgpe.icurr = 0
dgpe.inext = 1

dgpe.run_dynamics(no_pert=False)

Geometry:  (10, 10, 10)
Running scipy


In [7]:
# wSIM ∝ 1,
# wIBC ∝ exp(− lSIM ,)
# wPDE ∝ exp(− max(lSIM, lIBC),)

In [8]:
from sklearn.model_selection import train_test_split
import pandas as pd
X = step * np.arange(dgpe.X.shape[-1]).reshape(-1,1)
y = np.hstack((np.moveaxis(dgpe.X, -1, 0).reshape(-1, dgpe.N_wells), np.moveaxis(dgpe.Y, -1, 0).reshape(-1, dgpe.N_wells)))

train_size = int(0.75 * X.shape[0])
X_train_raw = X[:train_size, :]
y_train_raw = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]
X_train_raw.shape, X_test.shape

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.25, random_state=0xE2E4
)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((2812, 1), (938, 1), (2812, 2000), (938, 2000))

In [9]:
from src.dgpe_nn import DGPEModule
from src.pinn_lib import train_and_validate, plot_losses
from src.dataloaders import generate_datasets

In [ ]:
batch_size = 512
istride = 500
train_loader, test_loader, val_loader, init_loader = generate_datasets(X, y, X_train, y_train, X_test, y_test, X_val, y_val, batch_size, istride)

In [ ]:
from sklearn.preprocessing import LabelEncoder
n_hidden = 32

n_in, n_out = X.shape[-1], y.shape[-1]
model = DGPEModule(dgpe, n_in, n_out)

max_norm = 100.0  # Define a threshold
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.95, weight_decay=1e-3)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)#, momentum=0.95, weight_decay=1e-3)
def crit(x,y):
    print(x.shape, y.shape)
    return nn.MSELoss()(x,y)
# criterion = nn.MSELoss()
criterion = crit
metric = lambda x,y: nn.MSELoss()(x, y)
num_epochs = 30

wandb.watch(model, criterion, log="all", log_freq=10) # Log gradients and parameters

In [ ]:
from sklearn.preprocessing import LabelEncoder
n_hidden = 32

n_in, n_out = X.shape[-1], y.shape[-1]
model = DGPEModule(dgpe, n_in, n_out)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3,  weight_decay=1e-3)

criterion = nn.MSELoss()
metric = lambda x,y: nn.MSELoss()(x, y)
num_epochs = 30
wandb.watch(model, criterion, log="all", log_freq=10) # Log gradients and parameters

In [ ]:

optimizer = torch.optim.LBFGS(model.parameters(), lr=1e-3)

num_epochs = 100

train_and_validate(model, optimizer, criterion, metric, 
                    train_loader, val_loader, 
                    init_loader,
                    num_epochs,
                    criterion_init_cond=True,
                    w1=1., w2=1., w3=1., w4=1e-3, w5=1e-6, w6=1., w7=1.,
                   run=run, verbose=False
                    )

num_epochs = 100

train_and_validate(model, optimizer, criterion, metric, 
                    train_loader, val_loader, 
                    init_loader,
                    num_epochs,
                    criterion_init_cond=True,
                    criterion_ibound=True,
                    w1=1., w2=1., w3=1., w4=1e-3, w5=1e-6, w6=1., w7=1.,
                   run=run, verbose=False
                    )

num_epochs = 100

train_and_validate(model, optimizer, criterion, metric, 
                    train_loader, val_loader, 
                    init_loader,
                    num_epochs,
                    criterion_init_cond=True,
                    criterion_ibound=True,
                    criterion_Nconst=True,
                    w1=1, w2=1, w3=1, w4=1e-3, w5=1e-6, w6=1., w7=1.,
                   run=run, verbose=False
                    )


num_epochs = 100

train_and_validate(model, optimizer, criterion, metric, 
                    train_loader, val_loader, 
                    init_loader,
                    num_epochs,
                    criterion_init_cond=True,
                    criterion_ibound=True,
                    criterion_Nconst=True,
                    criterion_Econst=True,
                    w1=1, w2=1, w3=1, w4=1e-3, w5=1e-6, w6=1., w7=1.,
                   run=run, verbose=False
                    )
optimizer = torch.optim.LBFGS(model.parameters(), lr=1e-4)

num_epochs = 500

train_and_validate(model, optimizer, criterion, metric, 
                    train_loader, val_loader, 
                    init_loader,
                    num_epochs,
                    criterion_init_cond=True,
                    criterion_ibound=True,
                    criterion_Nconst=True,
                    criterion_Econst=True,
                    criterion_pinn=True,
                    w1=1, w2=1, w3=1, w4=1e-3, w5=1e-6, w6=1., w7=1.,
                   run=run, verbose=False
                    )

In [ ]:
for i_plot in np.random.randint(0, 2000, 5):
# for i_plot in [0]:
    print(i_plot)
    psi_000 = []
    psi_exp_000 = []
    for i in range(0, 5000):
        t = torch.tensor([0.001 * i], dtype=torch.float32, requires_grad=False)
        psi = model(t)
        psi_exp = torch.cat([torch.tensor(dgpe.X[:,:,:,i].flatten(), dtype=torch.float32), torch.tensor(dgpe.Y[:,:,:,i].flatten(), dtype=torch.float32)], dim=0)
        psi_000.append(psi[i_plot].detach().numpy())
        psi_exp_000.append(psi_exp[i_plot].detach().numpy())
    time = np.arange(5000) * 0.001
    plt.plot(time, psi_000, label='prediction')
    # plt.scatter(time[::istride], psi_000[::istride])
    plt.plot(time, psi_exp_000, color='r', label='simulation')
    plt.scatter(time[::istride], psi_exp_000[::istride], color='r')
    plt.axvline(0.75*5)
    plt.legend()
    plt.show()

dt step --> compress solution --> next dt step 
PINN --> difference
VAE quality
or
CNN — from t -> t + dt